# Load preTrained Models: Load and Evaluate

This notebook loads a pre-trained model and evaluates its predictions against training data. The workflow includes:

1. **Data Loading & Preprocessing**: Efficiently loads and preprocesses meteorological and ozone data from NetCDF files using a unified function
2. **Feature Selection**: Supports both daily and hourly frequency data with optional rasterized ozone channel
3. **Model Loading**: Loads the pre-trained model (with or without rasterized ozone features)
4. **Model Evaluation**: Evaluates model performance on the training dataset

The refactored code uses reusable functions for reproducibility and easier parameter tuning across different data frequencies and feature combinations.

# Imports

In [22]:
#import matplotlib.pyplot as plt
import warnings
# turn off futurewarning from iris
warnings.filterwarnings("ignore", category=FutureWarning)
# turn off netcdf unknown variable warning from iris
warnings.filterwarnings("ignore", category=UserWarning, message=".*NetCDF variable.*")
# turn off IrisCfNonSpanningVarWarning
warnings.filterwarnings("ignore", category=UserWarning, message=".*IrisCfNonSpanningVarWarning.*")

import numpy as np

from matplotlib import pyplot as plt

import tensorflow as tf

import iris
iris.FUTURE.save_split_attrs = True

from model_lib.ml_utils import (
    is_running_in_notebook,
    load_pretrained_model,
    load_and_preprocess_training_data,
    parse_arguments,
)


# User parameters

In [23]:
# User parameters
# Note: Adjust these parameters as needed before running the notebook

# model_type = "MLP" 
# model_type = "2DCNN" 
model_type = "3DCNN" 
# model_type = "CNN+LSTM" 
# model_type = "UNet" 
# model_type = "convLSTM" 

training_frequency = "daily"  # Set to "hourly" or "daily"
with_rasterized_ozone = False  # Set to True to include rasterized ozone features

ndays = 93 if training_frequency == "daily" else 30  # Number of days to load (93 for daily, 30 for hourly testing)

# Set number of k-folds for cross-validation (must be >=2)
k_folds = 5
print(f"\nUsing k-folds: {k_folds}. To change, set 'k_folds' to desired number of folds (must be >=2).")

# sequence_length is only a fallback for sequence models with variable-length temporal input.
# For fixed-length models, Cell 12 auto-detects and uses the model-declared temporal length.
if model_type in ["MLP", "2DCNN"]:
    sequence_length = 1
elif model_type in ["3DCNN", "CNN+LSTM", "convLSTM", "UNet"]:
    sequence_length = 3 if training_frequency == "daily" else 48
else:
    raise ValueError(
        "Unsupported model_type. Choose one of: "
        "MLP, 2DCNN, 3DCNN, CNN+LSTM, convLSTM, UNet"
    )

# Apply CLI arguments only when run from shell/script (skip in Jupyter notebook)
if not is_running_in_notebook():
    args = parse_arguments()
    training_frequency = args.training_frequency
    with_rasterized_ozone = args.with_rasterized_ozone
    model_type = args.model_type
    ndays = args.ndays
    sequence_length = args.sequence_length
    k_folds = args.k_folds

# config_name  (with_rasterized_ozone  or met_only) is used as key in results dict to store model, history, and tensors
config_name = "with_rasterized_ozone" if with_rasterized_ozone else "met_only"


Using k-folds: 5. To change, set 'k_folds' to desired number of folds (must be >=2).


# Loading data and Trained Model

## Load xtrain and ytrain

In [24]:
# This section will load the training data based on the specified frequency and preprocess it for use with the pretrained model.
# The feature_selection_list defines which meteorological features to include as input to the model. The target_selection_list defines the target variable(s) for prediction, which is ozone concentration in this case.
# The load_and_preprocess_training_data function will handle loading the data from NetCDF files, filtering the cubes based on the selected features and targets, handling any NaN values, and normalizing the data for use with the model.
# After loading and preprocessing the data, the pretrained model will be loaded from the specified path, and its architecture will be summarized.
# Finally, the model will be evaluated on the training data to check its performance before proceeding to test data evaluation.
# Make sure to adjust the parameters at the top of this section as needed before running the notebook, especially if you want to test with a different frequency or include/exclude rasterized ozone features.

# Shared meteorological features
feature_selection_list = [
    "air_pressure_at_sea_level",
    "air_temperature",
    "relative_humidity",
    "x_wind",
    "y_wind",
    "precipitation_amount",
    "photolysis_rate_of_nitrogen_dioxide",
]

# Add rasterized ozone channel only when requested
if with_rasterized_ozone:
    feature_selection_list.append("rasterized_aqum_O3_at_AURN_sites")

target_selection_list = ["mass_concentration_of_ozone_in_air"]
pretrained_model_path = "Trained_models"

# Load and preprocess training data using the unified function
print("\n" + "="*80)
print("LOADING AND PREPROCESSING TRAINING DATA")
print("="*80)

data = load_and_preprocess_training_data(
    training_frequency=training_frequency,
    ndays=ndays,
    feature_selection_list=feature_selection_list,
    target_selection_list=target_selection_list,
    verbose=False  # Set to True for detailed processing messages
)

# Extract results from the returned dictionary
xtrain_data_normalised = data['xtrain_data_normalised']
ytrain_data_normalised = data['ytrain_data_normalised']
feature_names = data['feature_names']
ntime = data['ntime']
nlat = data['nlat']
nlon = data['nlon']
nfeature = data['nfeature']
nfeature_multiplier = data['nfeature_multiplier']
xtrain_cube_list_regridded = data['xtrain_cube_list_regridded']
ytrain_cube_list = data['ytrain_cube_list']
ntarget = len(target_selection_list)

print("\n" + "="*80)
print("DATA LOADING COMPLETE")
print("="*80)
print(f"X-train shape: {xtrain_data_normalised.shape}")
print(f"Y-train shape: {ytrain_data_normalised.shape}")
print(f"Features ({nfeature} + {nfeature_multiplier-1} mask channels): {feature_names}")



LOADING AND PREPROCESSING TRAINING DATA
Loading training data from: processed_data/Daily_snapshots
Number of days to load: 93
Feature selection list: ['air_pressure_at_sea_level', 'air_temperature', 'relative_humidity', 'x_wind', 'y_wind', 'precipitation_amount', 'photolysis_rate_of_nitrogen_dioxide']
Target selection list: ['mass_concentration_of_ozone_in_air']
File multiplier based on frequency: 1
Expected number of files to load for Y-train: 93
Expected number of files to load for X-train: 93
Verbose mode is OFF: Detailed processing messages will not be printed.


DATA LOADING COMPLETE
X-train shape: (93, 112, 88, 7)
Y-train shape: (93, 112, 88, 1)
Features (7 + 0 mask channels): ['air_pressure_at_sea_level', 'air_temperature', 'photolysis_rate_of_nitrogen_dioxide', 'precipitation_amount', 'relative_humidity', 'x_wind', 'y_wind']


# load pretrained ML model

In [25]:

# Now that the training data is loaded and preprocessed, we can proceed to load the pretrained model and evaluate its performance on the training data before moving on to test data evaluation.
# The pretrained model will be loaded based on the specified parameters, and its architecture will be summarized to confirm that it has been loaded correctly.
# After loading the model, we will evaluate its performance on the training data to check for any issues before proceeding to test data evaluation.
# Make sure to adjust the parameters at the top of this section as needed before running the notebook, especially if you want to test with a different frequency or include/exclude rasterized ozone features.
# Note: The evaluation on training data is just a sanity check to ensure the model is working correctly with the loaded data. The main evaluation will be done on the test data in the next section.

# Load the pretrained model
print("\n" + "="*80)
print("LOADING PRETRAINED MODEL")
print("="*80)

if training_frequency not in {"daily", "hourly"}:
    raise ValueError("training_frequency must be 'daily' or 'hourly'")

# Support either spelling if present in the notebook
use_rasterized_ozone = globals().get(
    "with_rasterized_ozone",
    globals().get("with_rasterised_ozone", False)
)

model = load_pretrained_model(
    model_path=pretrained_model_path,
    model_type=model_type,
    frequency=training_frequency,
    with_rasterized_ozone=use_rasterized_ozone,
    kfold=k_folds,
)

model.summary()


LOADING PRETRAINED MODEL
Error loading model: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'Adam', 'config': {'name': 'adam', 'learning_rate': 8.099999831756577e-06, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}, 'loss': 'mse', 'loss_weights': None, 'metrics': ['mae'],

TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'Adam', 'config': {'name': 'adam', 'learning_rate': 8.099999831756577e-06, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}, 'loss': 'mse', 'loss_weights': None, 'metrics': ['mae'], 'weighted_metrics': None, 'run_eage

# Evaluation

In [ ]:
print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL ON TRAINING DATA")
print("=" * 80)

# Prepare evaluation tensors based on model input rank:
# rank 4 -> snapshot models (MLP, 2DCNN)
# rank 5 -> sequence models (3DCNN, CNN+LSTM, ConvLSTM, UNet)
model_input_shape = model.input_shape
input_rank = len(model_input_shape)

print(f"Model input shape: {model_input_shape}")
print(f"Model output shape: {model.output_shape}")
print(f"Raw X-train shape: {xtrain_data_normalised.shape}")
print(f"Raw Y-train shape: {ytrain_data_normalised.shape}")

if input_rank == 4:
    print("Detected snapshot model (no temporal axis in model input).")
    x_eval = xtrain_data_normalised.astype(np.float32)
    y_eval = ytrain_data_normalised.astype(np.float32)
    n_samples = x_eval.shape[0]
    sequence_length_eval = 1

elif input_rank == 5:
    print("Detected sequence model (temporal axis required in model input).")

    # Prefer the model-declared sequence length when fixed.
    model_sequence_length = model_input_shape[1]
    if model_sequence_length is None:
        sequence_length_eval = int(sequence_length)
        print(
            "Model accepts variable sequence length; "
            f"using sequence_length={sequence_length_eval} from parameters."
        )
    else:
        sequence_length_eval = int(model_sequence_length)
        print(f"Using sequence length from model input shape: {sequence_length_eval}")

    n_samples = ntime - sequence_length_eval + 1
    if n_samples <= 0:
        raise ValueError(
            f"sequence_length={sequence_length_eval} is too large for ntime={ntime}"
        )

    x_eval = np.array(
        [xtrain_data_normalised[i : i + sequence_length_eval] for i in range(n_samples)],
        dtype=np.float32,
    )
    y_eval = np.array(
        [ytrain_data_normalised[i + sequence_length_eval - 1] for i in range(n_samples)],
        dtype=np.float32,
    )

else:
    raise ValueError(
        f"Unsupported model input rank: {input_rank}. "
        f"Expected rank 4 or 5, got input_shape={model_input_shape}."
    )

print(f"Prepared X-eval shape: {x_eval.shape}")
print(f"Prepared Y-eval shape: {y_eval.shape}")
print(f"Evaluation samples: {n_samples}")

# Ensure data is convertible to tensors and evaluate the model
try:
    xtrain_tensor = tf.convert_to_tensor(x_eval, dtype=tf.float32)
    ytrain_tensor = tf.convert_to_tensor(y_eval, dtype=tf.float32)

    print(f"X-train tensor shape: {xtrain_tensor.shape}")
    print(f"Y-train tensor shape: {ytrain_tensor.shape}")

    expected_input_shape = tuple(model.input_shape[1:])
    actual_input_shape = tuple(xtrain_tensor.shape[1:])

    if actual_input_shape != expected_input_shape:
        raise ValueError(
            "Input shape mismatch for evaluation. "
            f"Expected {expected_input_shape}, got {actual_input_shape}."
        )

    train_loss = model.evaluate(xtrain_tensor, ytrain_tensor, verbose=1)
    print(f"\nTraining Loss: {train_loss}")

except (ValueError, TypeError, tf.errors.InvalidArgumentError) as e:
    print(f"Error during evaluation: {e}")
    print("Checking model and data compatibility...")
    print(f"Expected input shape: {model.input_shape}")
    print(f"Actual input shape: {tuple([None] + list(x_eval.shape[1:]))}")
    print("\nDebugging info:")
    print(f"  model_type: {model_type}")
    print(f"  input_rank: {input_rank}")
    print(f"  ntime: {ntime}")
    print(f"  sequence_length used for eval: {sequence_length_eval}")
    print(f"  n_samples: {n_samples}")
    print(f"  nfeature * nfeature_multiplier: {nfeature * nfeature_multiplier}")
    raise

In [ ]:
# If we're in a notebook, we can show detailed plots of predictions vs true values for the first few samples
if is_running_in_notebook():
    # PLOT PREDICTION for 5 steps against true values for the training data
    pred = model.predict(xtrain_tensor[:5])
    print(f"\nPredictions shape: {pred.shape}")
    print(f"True values shape: {ytrain_tensor[:5].shape}")
    residual = ytrain_tensor[:5] - pred
    print(f"Residual shape: {residual.shape}")

    # Relative error (residual / true value), guarded against divide-by-zero
    relative_error = tf.where(
        ytrain_tensor[:5] != 0,
        100 * (residual / ytrain_tensor[:5]),
        tf.zeros_like(residual),
    )
    print(f"Relative error shape: {relative_error.shape}")

    # Plot true vs predicted vs residual vs relative error for the first 5 samples
    print(f"\n=== Predictions: {config_name} with model {model_type} with k={k_folds} folds ===")
    for i in range(5):
        plt.figure(figsize=(20, 5))

        plt.subplot(1, 4, 1)
        plt.title(f"Sample {i + 1} - True Ozone")
        plt.suptitle(f"{config_name} with model {model_type} with k={k_folds} folds — Sample {i + 1}", fontsize=14)
        plt.imshow(ytrain_tensor[i, ::-1, :, 0], vmin=0, vmax=1, cmap='Spectral_r')
        plt.colorbar(label="Normalized Ozone Concentration")

        plt.subplot(1, 4, 2)
        plt.title(f"Sample {i + 1} - Predicted Ozone")
        plt.imshow(pred[i, ::-1, :, 0], vmin=0, vmax=1, cmap='Spectral_r')
        plt.colorbar(label="Normalized Ozone Concentration")

        plt.subplot(1, 4, 3)
        plt.title(f"Sample {i + 1} - Residual (True - Predicted)")
        plt.imshow(residual[i, ::-1, :, 0], vmin=-1, vmax=1, cmap='bwr')
        plt.colorbar(label="Residual Value")

        plt.subplot(1, 4, 4)
        plt.title(f"Sample {i + 1} - Relative Error [%]")
        plt.imshow(relative_error[i, ::-1, :, 0], vmin=-100, vmax=100, cmap='coolwarm')
        plt.colorbar(label="Relative Error [%]")

        plt.tight_layout()
        plt.show()